# MusIML Add-on Experiment — Multilingual Bengali VLM
## Kaggle-safe, auto-detect, dependency-fixed version

This version fixes both the Kaggle dependency issue and the BFF-15 class-folder spelling issue.

### Important before running
If you already ran the older notebook that upgraded `numpy`, `pandas`, `scipy`, or `scikit-learn`, **restart the Kaggle session first**. Then run this notebook from the top.

This notebook intentionally:

- does **not** install or import `sentence-transformers`;
- does **not** upgrade `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`, or Pillow;
- uses Jina CLIP v2 through the official `transformers.AutoModel` interface;
- implements the small statistical utilities needed here without SciPy/sklearn;
- auto-detects the BFF-15 and SylFishBD roots under `/kaggle/input`;
- validates the exact seven-class image counts before model inference.

**Primary question:** can a multilingual VLM recover useful discrimination from Bengali-script fish names where BioCLIP2 was near chance?

**V2 fix:** the BFF-15 mount uses a Tilapia folder spelling that was not covered by the first detector. The alias set now includes `Tilapia`, `Telapia`, `Telapiya`, and `Tilapiya` variants, and root detection scans each attached dataset only once.


## 0. Minimal setup

This installs only the packages needed by Jina CLIP v2's direct Transformers implementation. It deliberately leaves Kaggle's scientific Python stack untouched.

If the previous broken notebook was run in this same session, use Kaggle's **Restart Session** first.

In [ ]:
%pip -q install "transformers>=4.44,<5" "timm>=1.0,<2" "einops>=0.8,<1" "safetensors>=0.4"

In [ ]:
from pathlib import Path
import os, json, random, re, unicodedata, platform, math
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import torch
from transformers import AutoModel

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Experiment configuration

For Kaggle, leave both root overrides as `None`. The detector will search `/kaggle/input`, rank candidate dataset roots by the exact per-class counts from the manuscript, and print the selected paths before inference.

Verify the seven Bengali strings with a native Bangladeshi Bengali speaker, then set `BENGALI_LABELS_VERIFIED = True`.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")

BFF_ROOT_OVERRIDE = None
SYL_ROOT_OVERRIDE = None

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_DIR = WORK_ROOT / "musiml_multilingual_results"
CACHE_DIR = OUTPUT_DIR / "embedding_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "jinaai/jina-clip-v2"
TRUNCATE_DIM = 512
IMAGE_BATCH_SIZE = 6
BOOTSTRAP_RESAMPLES = 2000

STRICT_EXPECTED_COUNTS = True
SMOKE_TEST = False
SMOKE_N_PER_CLASS = 10

CLASSES = ["Rui", "Katla", "Mrigal", "Tilapia", "Pabda", "Ilish", "Koi"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

EXPECTED_COUNTS = {
    "BFF-15": {
        "Rui": 514, "Katla": 427, "Mrigal": 317, "Tilapia": 383,
        "Pabda": 348, "Ilish": 233, "Koi": 434,
    },
    "SylFishBD": {
        "Rui": 1670, "Katla": 1133, "Mrigal": 1293, "Tilapia": 1326,
        "Pabda": 862, "Ilish": 789, "Koi": 592,
    },
}

CLASS_ALIASES = {
    "Rui": ["rui", "rohu", "labeo rohita", "labeo_rohita"],
    "Katla": ["katla", "catla", "catla catla", "catla_catla"],
    "Mrigal": ["mrigal", "mrigel", "mrigal carp", "cirrhinus cirrhosus", "cirrhinus_cirrhosus"],
    "Tilapia": [
        "tilapia", "tilapia fish",
        "telapia", "telapia fish",
        "telapiya", "telapiya fish",
        "tilapiya", "tilapiya fish",
        "nile tilapia",
        "oreochromis niloticus", "oreochromis_niloticus"
    ],
    "Pabda": ["pabda", "pabda catfish", "ompok pabda", "ompok_pabda"],
    "Ilish": ["ilish", "hilsa", "tenualosa ilisha", "tenualosa_ilisha"],
    "Koi": ["koi", "climbing perch", "anabas testudineus", "anabas_testudineus"],
}

NAMES = {
    "bengali": {
        "Rui": "রুই", "Katla": "কাতলা", "Mrigal": "মৃগেল",
        "Tilapia": "তেলাপিয়া", "Pabda": "পাবদা", "Ilish": "ইলিশ", "Koi": "কৈ",
    },
    "romanized": {
        "Rui": "Rui", "Katla": "Katla", "Mrigal": "Mrigal",
        "Tilapia": "Tilapia", "Pabda": "Pabda", "Ilish": "Ilish", "Koi": "Koi",
    },
    "english": {
        "Rui": "Rohu", "Katla": "Catla", "Mrigal": "Mrigal carp",
        "Tilapia": "Nile tilapia", "Pabda": "Pabda catfish",
        "Ilish": "Hilsa", "Koi": "Climbing perch",
    },
    "scientific": {
        "Rui": "Labeo rohita", "Katla": "Catla catla",
        "Mrigal": "Cirrhinus cirrhosus", "Tilapia": "Oreochromis niloticus",
        "Pabda": "Ompok pabda", "Ilish": "Tenualosa ilisha",
        "Koi": "Anabas testudineus",
    },
}

BENGALI_LABELS_VERIFIED = False

TEMPLATES = [
    "a photo of {name}, a fish species",
    "an image of {name}, a fish species",
    "a photograph of {name}, a fish species",
    "a specimen of {name}, a fish species",
]

print("Output:", OUTPUT_DIR)
print("\nBengali labels to verify:")
for c in CLASSES:
    print(f"{c:8s} -> {NAMES['bengali'][c]}")

## 2. Kaggle input auto-detection — optimized

The previous run found **2,273 / 2,656 BFF-15 images**. The difference is exactly **383 images**, which is the paper's Tilapia count. That indicates a folder-label spelling mismatch, not missing data.

This version therefore:

1. recognizes `Tilapia`, `Telapia`, `Telapiya`, and `Tilapiya` folder spellings;
2. scans each attached dataset root **once** instead of recursively rescanning many candidate subfolders;
3. chooses the BFF-15 and SylFishBD mounts by distance to the exact per-class counts;
4. refines each selected mount to the deepest common folder containing the matched images;
5. prints plain-text per-class counts so Kaggle version logs remain readable.

Expected totals: **2,656 BFF-15** and **7,665 SylFishBD**.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

EXCLUDE_COMPONENTS = {
    "mask", "masks", "segmentation", "segmentations",
    "annotation", "annotations", "bbox", "bboxes",
    "bounding box", "bounding boxes",
}

def norm_token(x):
    x = unicodedata.normalize("NFKC", str(x)).lower().strip()
    return re.sub(r"[\s\-_]+", " ", x)

ALIAS_TO_CLASS = {}
for cls, aliases in CLASS_ALIASES.items():
    ALIAS_TO_CLASS[norm_token(cls)] = cls
    for alias in aliases:
        ALIAS_TO_CLASS[norm_token(alias)] = cls

def excluded_path(path):
    for part in Path(path).parts:
        token = norm_token(part)
        if token in EXCLUDE_COMPONENTS:
            return True
        if any(
            token.startswith(x + " ") or token.endswith(" " + x)
            for x in EXCLUDE_COMPONENTS
        ):
            return True
    return False

def infer_class(path):
    p = Path(path)
    tokens = [norm_token(x) for x in list(p.parts) + [p.stem]]

    # Exact path-component match first.
    for token in reversed(tokens):
        if token in ALIAS_TO_CLASS:
            return ALIAS_TO_CLASS[token]

    # Conservative whole-token fallback.
    for token in reversed(tokens):
        for alias, cls in ALIAS_TO_CLASS.items():
            if len(alias) >= 3 and re.search(
                rf"(^| ){re.escape(alias)}($| )", token
            ):
                return cls
    return None

def build_manifest(root, dataset_name):
    root = Path(root)
    rows = []

    if not root.exists():
        return pd.DataFrame(columns=["dataset", "path", "class", "y"])

    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:
            continue
        if excluded_path(p):
            continue

        cls = infer_class(p)
        if cls is not None:
            rows.append({
                "dataset": dataset_name,
                "path": str(p),
                "class": cls,
                "y": CLASS_TO_ID[cls],
            })

    if not rows:
        return pd.DataFrame(columns=["dataset", "path", "class", "y"])

    return (
        pd.DataFrame(rows)
        .drop_duplicates("path")
        .sort_values(["y", "path"])
        .reset_index(drop=True)
    )

def class_counts(df):
    vc = df["class"].value_counts() if len(df) else pd.Series(dtype=int)
    return {c: int(vc.get(c, 0)) for c in CLASSES}

def print_counts(df, label):
    cnt = class_counts(df)
    print(f"\n{label}: {len(df)} matched target images")
    for c in CLASSES:
        print(f"  {c:8s}: {cnt[c]}")

def distance_from_expected(df, dataset_name):
    got = class_counts(df)
    exp = EXPECTED_COUNTS[dataset_name]
    return sum(abs(got[c] - exp[c]) for c in CLASSES)

def attached_dataset_roots():
    """
    Handles both:
      /kaggle/input/<dataset-slug>
    and the layout seen in this notebook:
      /kaggle/input/datasets/<owner>/...
    """
    if not KAGGLE_INPUT.exists():
        raise FileNotFoundError(
            "/kaggle/input not found. Attach the two Kaggle datasets first."
        )

    nested = KAGGLE_INPUT / "datasets"

    if nested.exists() and nested.is_dir():
        roots = [p for p in nested.iterdir() if p.is_dir()]
        container = nested
    else:
        roots = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
        container = KAGGLE_INPUT

    print("Dataset container:", container)
    print("Attached dataset roots:")
    for r in roots:
        print("  ", r)

    if not roots:
        raise RuntimeError("No attached dataset directories found.")

    return roots

def deepest_common_root(df, fallback):
    """
    Refine a broad Kaggle owner/mount directory to the common parent
    containing all matched target-class images.
    """
    if len(df) == 0:
        return Path(fallback)

    try:
        common = Path(os.path.commonpath(df["path"].tolist()))
    except Exception:
        return Path(fallback)

    if common.is_file():
        common = common.parent

    # Do not return a path outside the selected broad root.
    try:
        common.relative_to(Path(fallback))
    except Exception:
        return Path(fallback)

    return common

def detect_roots():
    records = []
    cached = {}

    # Each attached root is recursively scanned exactly once.
    for root in tqdm(attached_dataset_roots(), desc="Scanning attached datasets"):
        df = build_manifest(root, "candidate")
        cached[str(root)] = df

        if len(df) == 0:
            continue

        cnt = class_counts(df)
        coverage = sum(cnt[c] > 0 for c in CLASSES)

        records.append({
            "root": str(root),
            "coverage": coverage,
            "total_target_images": len(df),
            "bff_distance": distance_from_expected(df, "BFF-15"),
            "syl_distance": distance_from_expected(df, "SylFishBD"),
            **{f"n_{c}": cnt[c] for c in CLASSES},
        })

    if not records:
        raise RuntimeError(
            "No target fish classes were found under the attached Kaggle inputs."
        )

    report = pd.DataFrame(records)

    bff_rank = report.sort_values(
        ["bff_distance", "coverage"],
        ascending=[True, False],
    )
    syl_rank = report.sort_values(
        ["syl_distance", "coverage"],
        ascending=[True, False],
    )

    broad_bff = Path(bff_rank.iloc[0]["root"])
    broad_syl = Path(syl_rank.iloc[0]["root"])

    bff_df = cached[str(broad_bff)]
    syl_df = cached[str(broad_syl)]

    refined_bff = deepest_common_root(bff_df, broad_bff)
    refined_syl = deepest_common_root(syl_df, broad_syl)

    return refined_bff, refined_syl, bff_rank, syl_rank

if BFF_ROOT_OVERRIDE is None or SYL_ROOT_OVERRIDE is None:
    auto_bff, auto_syl, bff_rank, syl_rank = detect_roots()
else:
    auto_bff = Path(BFF_ROOT_OVERRIDE)
    auto_syl = Path(SYL_ROOT_OVERRIDE)
    bff_rank = syl_rank = None

BFF_ROOT = (
    Path(BFF_ROOT_OVERRIDE)
    if BFF_ROOT_OVERRIDE is not None
    else auto_bff
)
SYL_ROOT = (
    Path(SYL_ROOT_OVERRIDE)
    if SYL_ROOT_OVERRIDE is not None
    else auto_syl
)

print("\nSelected roots")
print("BFF-15   :", BFF_ROOT)
print("SylFishBD:", SYL_ROOT)

if bff_rank is not None:
    print("\nBFF ranking:")
    print(
        bff_rank[
            ["root", "coverage", "total_target_images", "bff_distance"]
            + [f"n_{c}" for c in CLASSES]
        ].to_string(index=False)
    )

    print("\nSylFishBD ranking:")
    print(
        syl_rank[
            ["root", "coverage", "total_target_images", "syl_distance"]
            + [f"n_{c}" for c in CLASSES]
        ].to_string(index=False)
    )

## 3. Final count validation

In [ ]:
bff = build_manifest(BFF_ROOT, "BFF-15")
syl = build_manifest(SYL_ROOT, "SylFishBD")

def unknown_parent_folders(root, top_n=30):
    """
    If validation fails, show image parent folders that were not mapped
    to any of the seven classes. This makes spelling mismatches obvious.
    """
    counts = {}

    for p in Path(root).rglob("*"):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:
            continue
        if excluded_path(p):
            continue
        if infer_class(p) is None:
            parent = str(p.parent)
            counts[parent] = counts.get(parent, 0) + 1

    return sorted(counts.items(), key=lambda x: x[1], reverse=True)[:top_n]

def validate_counts(df, dataset_name, root):
    got = class_counts(df)
    exp = EXPECTED_COUNTS[dataset_name]

    table = pd.DataFrame({
        "class": CLASSES,
        "expected": [exp[c] for c in CLASSES],
        "found": [got[c] for c in CLASSES],
    })
    table["difference"] = table["found"] - table["expected"]
    table["match"] = table["expected"] == table["found"]

    print(f"\n===== {dataset_name} VALIDATION =====")
    print(table.to_string(index=False))
    print("Expected total:", int(table["expected"].sum()))
    print("Found total:   ", int(table["found"].sum()))

    if STRICT_EXPECTED_COUNTS and not table["match"].all():
        print("\nLargest unmapped image folders under the selected root:")
        unknown = unknown_parent_folders(root)
        if unknown:
            for folder, n in unknown:
                print(f"  {n:5d}  {folder}")
        else:
            print("  None found.")

        raise AssertionError(
            f"{dataset_name} count mismatch. The plain-text table above "
            "shows exactly which class differs."
        )

validate_counts(bff, "BFF-15", BFF_ROOT)
validate_counts(syl, "SylFishBD", SYL_ROOT)

if SMOKE_TEST:
    bff = (
        bff.groupby("class", group_keys=False)
        .head(SMOKE_N_PER_CLASS)
        .reset_index(drop=True)
    )
    syl = (
        syl.groupby("class", group_keys=False)
        .head(SMOKE_N_PER_CLASS)
        .reset_index(drop=True)
    )

assert BENGALI_LABELS_VERIFIED, (
    "Dataset detection succeeded. Verify the Bengali labels, then set "
    "BENGALI_LABELS_VERIFIED=True in the configuration cell."
)

## 4. Load Jina CLIP v2 directly through Transformers

This follows the model's documented direct `AutoModel` route and avoids `sentence-transformers`.

In [ ]:
print("Loading:", MODEL_ID)

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
model = model.to(DEVICE)
model.eval()

print("Model loaded on", DEVICE)

## 5. Compatibility smoke test

In [ ]:
def as_numpy(x):
    if isinstance(x, np.ndarray):
        return x
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)

def l2_normalize(x):
    x = as_numpy(x).astype(np.float32)
    if x.ndim == 1:
        x = x[None, :]
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

test_text = model.encode_text(
    ["a photo of Rohu, a fish species"],
    truncate_dim=TRUNCATE_DIM,
)

with Image.open(bff.iloc[0]["path"]) as im:
    test_image = model.encode_image(
        [im.convert("RGB").copy()],
        truncate_dim=TRUNCATE_DIM,
    )

test_text = l2_normalize(test_text)
test_image = l2_normalize(test_image)

print("Text shape :", test_text.shape)
print("Image shape:", test_image.shape)
print("Cosine similarity:", float(test_text[0] @ test_image[0]))
assert test_text.shape[1] == test_image.shape[1]

## 6. Cache image embeddings

In [ ]:
def safe_slug(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)

def encode_dataset_images(df, dataset_name, force=False):
    cache_path = CACHE_DIR / f"{safe_slug(dataset_name)}__jina_clip_v2__d{TRUNCATE_DIM}.npz"
    paths = df["path"].tolist()
    labels = df["y"].to_numpy(np.int64)

    if cache_path.exists() and not force:
        z = np.load(cache_path, allow_pickle=True)
        if z["paths"].tolist() != paths:
            raise RuntimeError("Cache path order does not match current manifest.")
        if not np.array_equal(z["y"], labels):
            raise RuntimeError("Cache labels do not match current manifest.")
        print("Loaded cache:", cache_path)
        return z["emb"].astype(np.float32)

    chunks = []

    for start in tqdm(range(0, len(paths), IMAGE_BATCH_SIZE), desc=f"Encoding {dataset_name}"):
        batch_paths = paths[start:start + IMAGE_BATCH_SIZE]
        images = []

        for p in batch_paths:
            with Image.open(p) as im:
                images.append(im.convert("RGB").copy())

        with torch.inference_mode():
            emb = model.encode_image(images, truncate_dim=TRUNCATE_DIM)

        chunks.append(l2_normalize(emb))

    emb = np.concatenate(chunks, axis=0).astype(np.float32)

    np.savez_compressed(
        cache_path,
        emb=emb,
        paths=np.array(paths, dtype=object),
        y=labels,
    )

    print("Saved:", cache_path, emb.shape)
    return emb

bff_img = encode_dataset_images(bff, "BFF-15")
syl_img = encode_dataset_images(syl, "SylFishBD")

## 7. Text prototypes and zero-shot predictions

In [ ]:
FAMILIES = ["bengali", "romanized", "english", "scientific"]

def make_prototypes(family, protocol="paper_4template"):
    if protocol == "paper_4template":
        groups = [
            [t.format(name=NAMES[family][cls]) for t in TEMPLATES]
            for cls in CLASSES
        ]
    elif protocol == "name_only":
        groups = [[NAMES[family][cls]] for cls in CLASSES]
    else:
        raise ValueError(protocol)

    flat = [p for group in groups for p in group]

    with torch.inference_mode():
        emb = model.encode_text(flat, truncate_dim=TRUNCATE_DIM)

    emb = l2_normalize(emb)
    k = len(groups[0])
    emb = emb.reshape(len(CLASSES), k, -1)

    proto = emb.mean(axis=1)
    return l2_normalize(proto)

def run_protocol(df, image_emb, protocol):
    out = df.copy()

    for family in FAMILIES:
        proto = make_prototypes(family, protocol)
        scores = image_emb @ proto.T
        pred = scores.argmax(axis=1).astype(np.int64)

        out[f"pred__{family}"] = pred
        out[f"correct__{family}"] = pred == out["y"].to_numpy()

    return out

primary_bff = run_protocol(bff, bff_img, "paper_4template")
primary_syl = run_protocol(syl, syl_img, "paper_4template")
nameonly_bff = run_protocol(bff, bff_img, "name_only")
nameonly_syl = run_protocol(syl, syl_img, "name_only")

print("Prediction complete.")

## 8. Accuracy, balanced accuracy, macro-F1, bootstrap 95% CIs

In [ ]:
def metric_triplet(y, pred):
    y = np.asarray(y, dtype=np.int64)
    pred = np.asarray(pred, dtype=np.int64)

    accuracy = np.mean(y == pred)
    recalls = []
    f1s = []

    for c in range(len(CLASSES)):
        tp = np.sum((y == c) & (pred == c))
        fn = np.sum((y == c) & (pred != c))
        fp = np.sum((y != c) & (pred == c))

        recall = tp / (tp + fn) if tp + fn else 0.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        recalls.append(recall)
        f1s.append(f1)

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1s)),
    }

def bootstrap_metrics(y, pred, B=BOOTSTRAP_RESAMPLES, seed=SEED):
    y = np.asarray(y, dtype=np.int64)
    pred = np.asarray(pred, dtype=np.int64)
    rng = np.random.default_rng(seed)
    idx_by_class = [np.flatnonzero(y == c) for c in range(len(CLASSES))]
    vals = np.empty((B, 3), dtype=np.float32)

    for b in range(B):
        idx = np.concatenate([
            rng.choice(ix, size=len(ix), replace=True)
            for ix in idx_by_class
        ])
        m = metric_triplet(y[idx], pred[idx])
        vals[b] = [m["accuracy"], m["balanced_accuracy"], m["macro_f1"]]

    return np.quantile(vals, [0.025, 0.975], axis=0)

def summarize(df, dataset, protocol):
    y = df["y"].to_numpy()
    rows = []

    for family in FAMILIES:
        pred = df[f"pred__{family}"].to_numpy()
        m = metric_triplet(y, pred)
        ci = bootstrap_metrics(y, pred)

        rows.append({
            "dataset": dataset,
            "protocol": protocol,
            "family": family,
            "n": len(y),
            "accuracy_pct": 100 * m["accuracy"],
            "accuracy_ci_low_pct": 100 * ci[0, 0],
            "accuracy_ci_high_pct": 100 * ci[1, 0],
            "balanced_accuracy_pct": 100 * m["balanced_accuracy"],
            "bacc_ci_low_pct": 100 * ci[0, 1],
            "bacc_ci_high_pct": 100 * ci[1, 1],
            "macro_f1_pct": 100 * m["macro_f1"],
            "f1_ci_low_pct": 100 * ci[0, 2],
            "f1_ci_high_pct": 100 * ci[1, 2],
        })

    return pd.DataFrame(rows)

results = pd.concat([
    summarize(primary_bff, "BFF-15", "paper_4template"),
    summarize(primary_syl, "SylFishBD", "paper_4template"),
    summarize(nameonly_bff, "BFF-15", "name_only"),
    summarize(nameonly_syl, "SylFishBD", "name_only"),
], ignore_index=True)

display(results.round(2))

## 9. Exact paired McNemar tests + Benjamini-Hochberg FDR

In [ ]:
PRIMARY_PAIRS = [
    ("bengali", "romanized"),
    ("bengali", "english"),
    ("romanized", "english"),
]

def log_choose(n, k):
    return math.lgamma(n + 1) - math.lgamma(k + 1) - math.lgamma(n - k + 1)

def exact_two_sided_binomial_half(k, n):
    if n == 0:
        return 1.0

    m = min(k, n - k)
    logs = np.array(
        [log_choose(n, i) - n * math.log(2.0) for i in range(m + 1)],
        dtype=np.float64,
    )

    mx = float(logs.max())
    lower = math.exp(mx) * float(np.exp(logs - mx).sum())
    return min(1.0, 2.0 * lower)

def exact_mcnemar(correct_a, correct_b):
    a = np.asarray(correct_a, dtype=bool)
    b = np.asarray(correct_b, dtype=bool)

    n01 = int(np.sum(~a & b))
    n10 = int(np.sum(a & ~b))
    n = n01 + n10

    return n01, n10, exact_two_sided_binomial_half(n01, n)

def paired_bootstrap_delta(y, ca, cb, B=BOOTSTRAP_RESAMPLES, seed=SEED):
    y = np.asarray(y, dtype=np.int64)
    ca = np.asarray(ca, dtype=bool)
    cb = np.asarray(cb, dtype=bool)

    rng = np.random.default_rng(seed)
    idx_by_class = [np.flatnonzero(y == c) for c in range(len(CLASSES))]
    values = np.empty(B, dtype=np.float32)

    for b in range(B):
        idx = np.concatenate([
            rng.choice(ix, size=len(ix), replace=True)
            for ix in idx_by_class
        ])
        values[b] = cb[idx].mean() - ca[idx].mean()

    return np.quantile(values, [0.025, 0.975])

def bh_fdr(pvalues):
    p = np.asarray(pvalues, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)

    out = np.empty_like(q)
    out[order] = q
    return out

def language_tests(df, dataset):
    y = df["y"].to_numpy()
    rows = []

    for a, b in PRIMARY_PAIRS:
        ca = df[f"correct__{a}"].to_numpy(bool)
        cb = df[f"correct__{b}"].to_numpy(bool)

        lo, hi = paired_bootstrap_delta(y, ca, cb)
        n01, n10, p = exact_mcnemar(ca, cb)

        rows.append({
            "dataset": dataset,
            "comparison": f"{a} -> {b}",
            "acc_A_pct": 100 * ca.mean(),
            "acc_B_pct": 100 * cb.mean(),
            "delta_B_minus_A_pp": 100 * (cb.mean() - ca.mean()),
            "ci_low_pp": 100 * lo,
            "ci_high_pp": 100 * hi,
            "A_wrong_B_right": n01,
            "A_right_B_wrong": n10,
            "mcnemar_exact_p": p,
        })

    return pd.DataFrame(rows)

tests = pd.concat([
    language_tests(primary_bff, "BFF-15"),
    language_tests(primary_syl, "SylFishBD"),
], ignore_index=True)

tests["fdr_q"] = bh_fdr(tests["mcnemar_exact_p"].to_numpy())
display(tests.round(5))

## 10. Per-class recall and confusion matrices

In [ ]:
def per_class_recall(df, dataset):
    y = df["y"].to_numpy()
    rows = []

    for family in FAMILIES:
        pred = df[f"pred__{family}"].to_numpy()

        for c, cls in enumerate(CLASSES):
            mask = y == c
            rows.append({
                "dataset": dataset,
                "family": family,
                "class": cls,
                "n": int(mask.sum()),
                "recall_pct": 100 * np.mean(pred[mask] == c),
            })

    return pd.DataFrame(rows)

per_class = pd.concat([
    per_class_recall(primary_bff, "BFF-15"),
    per_class_recall(primary_syl, "SylFishBD"),
], ignore_index=True)

display(
    per_class.pivot_table(
        index=["dataset", "class"],
        columns="family",
        values="recall_pct",
    ).round(2)
)

def confusion_numpy(y, pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y, pred):
        cm[int(t), int(p)] += 1
    return cm

def save_confusion_csv(df, dataset, family="bengali"):
    y = df["y"].to_numpy()
    pred = df[f"pred__{family}"].to_numpy()
    cm = confusion_numpy(y, pred, len(CLASSES))

    out_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
    path = OUTPUT_DIR / f"confusion_{dataset}_{family}.csv"
    out_df.to_csv(path)
    print("Saved:", path)
    display(out_df)

save_confusion_csv(primary_bff, "BFF-15")
save_confusion_csv(primary_syl, "SylFishBD")

## 11. Descriptive comparison with the current paper's BioCLIP2 results

In [ ]:
BIOCLIP2_REPORTED = pd.DataFrame([
    {"dataset": "BFF-15", "family": "bengali", "bioclip2_accuracy_pct": 16.27, "bioclip2_balanced_accuracy_pct": 14.22},
    {"dataset": "SylFishBD", "family": "bengali", "bioclip2_accuracy_pct": 7.74, "bioclip2_balanced_accuracy_pct": 14.29},
    {"dataset": "BFF-15", "family": "romanized", "bioclip2_accuracy_pct": 35.77, "bioclip2_balanced_accuracy_pct": np.nan},
    {"dataset": "SylFishBD", "family": "romanized", "bioclip2_accuracy_pct": 37.56, "bioclip2_balanced_accuracy_pct": np.nan},
    {"dataset": "BFF-15", "family": "english", "bioclip2_accuracy_pct": 72.36, "bioclip2_balanced_accuracy_pct": np.nan},
    {"dataset": "SylFishBD", "family": "english", "bioclip2_accuracy_pct": 64.59, "bioclip2_balanced_accuracy_pct": np.nan},
])

primary_metrics = results[results["protocol"] == "paper_4template"][
    ["dataset", "family", "accuracy_pct", "balanced_accuracy_pct", "macro_f1_pct"]
].rename(columns={
    "accuracy_pct": "jina_accuracy_pct",
    "balanced_accuracy_pct": "jina_balanced_accuracy_pct",
    "macro_f1_pct": "jina_macro_f1_pct",
})

comparison = BIOCLIP2_REPORTED.merge(
    primary_metrics,
    on=["dataset", "family"],
    how="outer",
)

display(comparison.round(2))

## 12. Save paper-ready outputs

In [ ]:
results.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
tests.to_csv(OUTPUT_DIR / "pairwise_language_tests.csv", index=False)
per_class.to_csv(OUTPUT_DIR / "per_class_recall.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "jina_vs_paper_reported_bioclip2.csv", index=False)

primary_bff.to_csv(OUTPUT_DIR / "predictions_BFF15_jina_4template.csv", index=False)
primary_syl.to_csv(OUTPUT_DIR / "predictions_SylFishBD_jina_4template.csv", index=False)
nameonly_bff.to_csv(OUTPUT_DIR / "predictions_BFF15_jina_name_only.csv", index=False)
nameonly_syl.to_csv(OUTPUT_DIR / "predictions_SylFishBD_jina_name_only.csv", index=False)

table = results[results["protocol"] == "paper_4template"][
    ["dataset", "family", "accuracy_pct", "balanced_accuracy_pct", "macro_f1_pct"]
].sort_values(["dataset", "family"])

try:
    latex = table.to_latex(
        index=False,
        float_format=lambda x: f"{x:.2f}",
        caption="Zero-shot Jina CLIP v2 results under the predefined four-template ensemble.",
        label="tab:jina_multilingual",
    )
    (OUTPUT_DIR / "jina_multilingual_table.tex").write_text(latex, encoding="utf-8")
    print(latex)
except Exception as e:
    print("LaTeX export skipped:", repr(e))

print("\nSaved to:", OUTPUT_DIR)

## 13. Reproducibility snapshot and manuscript helper

In [ ]:
import transformers
import timm
import einops

snapshot = {
    "seed": SEED,
    "model_id": MODEL_ID,
    "truncate_dim": TRUNCATE_DIM,
    "device": DEVICE,
    "image_batch_size": IMAGE_BATCH_SIZE,
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "dataset_roots": {
        "BFF-15": str(BFF_ROOT),
        "SylFishBD": str(SYL_ROOT),
    },
    "dataset_sizes": {
        "BFF-15": len(bff),
        "SylFishBD": len(syl),
    },
    "templates": TEMPLATES,
    "names": NAMES,
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "timm": timm.__version__,
        "einops": einops.__version__,
    },
}

(OUTPUT_DIR / "reproducibility_snapshot.json").write_text(
    json.dumps(snapshot, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(snapshot, ensure_ascii=False, indent=2))

# FIX: use `results` here. `primary_metrics` renamed these columns
# with a `jina_` prefix for the descriptive BioCLIP2 comparison.
paper_metrics = (
    results[results["protocol"] == "paper_4template"]
    .copy()
    .set_index(["dataset", "family"])
)

def manuscript_sentence(dataset):
    b = paper_metrics.loc[(dataset, "bengali")]
    r = paper_metrics.loc[(dataset, "romanized")]
    e = paper_metrics.loc[(dataset, "english")]

    t = tests[
        (tests["dataset"] == dataset)
        & (tests["comparison"] == "bengali -> english")
    ].iloc[0]

    return (
        f"{dataset}: Jina CLIP v2 achieved "
        f"{b['balanced_accuracy_pct']:.2f}% balanced accuracy "
        f"with Bengali-script names, versus "
        f"{r['balanced_accuracy_pct']:.2f}% Romanized and "
        f"{e['balanced_accuracy_pct']:.2f}% English. "
        f"English-minus-Bengali accuracy difference: "
        f"{t['delta_B_minus_A_pp']:.2f} pp "
        f"(95% CI [{t['ci_low_pp']:.2f}, {t['ci_high_pp']:.2f}], "
        f"FDR q={t['fdr_q']:.4g})."
    )

print("\nPAPER HELPER")
print(manuscript_sentence("BFF-15"))
print(manuscript_sentence("SylFishBD"))

# Interpretation rule for the MusIML paper

Use the four-template results as primary.

- If Bengali improves materially while English/Romanized are also useful, the BioCLIP2 Bengali failure is at least partly a multilingual alignment/interface issue.
- If Bengali remains near chance but Jina English/Romanized work, explicit multilingual support still does not guarantee reliable fine-grained Bengali biological nomenclature.
- If all Jina conditions perform poorly, Jina lacks sufficient fine-grained fish discrimination here, so do not make a strong Bengali-specific conclusion.
- Treat `name_only` only as a secondary prompt-frame control.
- Keep claims limited to these seven classes and two sources.